# 과제 2 - 분류
### churn_train.csv 를 이용하여 churn_test.csv 의 churn 을 분류

#### churn_dataspec(데이터명세서) 참고
- 타깃(클래스)
- Churn : 1=이탈, 0=유지
- 특징(입력 변수)
- Age : 나이
- Tenure_Months : 가입기간(개월)
- Monthly_Fee : 월 요금
- Total_Usage_GB : 최근 사용량(GB)
- Support_Tickets_3M : 최근 3개월 CS 문의 건수
- Late_Payments_6M : 최근 6개월 연체 횟수
- Contract : 계약 형태(Month-to-month / 1-year / 2-year)
- AutoPay : 자동결제(0/1)
- Internet_Type : 회선(Fiber/DSL/5G)
- Has_Addon : 부가서비스(0/1)
- NPS_Score : 만족도 점수(-100~100)
- Region : 지역(Seoul/Metro/Other)

## Plan: Churn 분류 실습 파이프라인 (DRAFT)

요청하신 데이터 기준으로, [ml-day05-013-HW2-chr-cla.ipynb](ml-day05-013-HW2-chr-cla.ipynb)에서 재현 가능한 분류 파이프라인을 단계적으로 구성합니다. 
확정된 의사결정은 Accuracy를 1순위 지표로 사용하고, 
모델 선택은 Stratified 5-Fold CV로 수행하며, 
[data/churn_test.csv](data/churn_test.csv)의 Churn은 학습/모델선정에 사용하지 않고 최종 사후 점검에만 활용하는 것입니다. 
기존 레포의 실습 관례(전처리+모델 Pipeline, 분류 리포트 출력, 결과 csv 저장)를 유지해 과제 스타일과 일관성을 맞추고, 
누수 방지와 재현성(random_state 고정)을 기본 원칙으로 둡니다.

**Steps**
1. 데이터 스키마 고정: [data/churn_train.csv](data/churn_train.csv), [data/churn_test.csv](data/churn_test.csv), [data/churn_dataspec](data/churn_dataspec)을 기준으로 타깃(Churn), 식별자(Customer_ID), 수치형/범주형 컬럼 목록을 명시합니다.
2. EDA 최소 점검: [ml-day05-013-HW2-chr-cla.ipynb](ml-day05-013-HW2-chr-cla.ipynb)에서 클래스 분포, 결측치, 기초 기술통계를 확인하고 불균형 정도를 기록합니다.
3. 피처/타깃 분리: 학습 데이터에서 Customer_ID는 모델 입력에서 제외하고, Churn을 y로 분리해 누수 가능성을 차단합니다.
4. 전처리 파이프라인 구성: ColumnTransformer로 수치형(필요 시 스케일링)과 범주형(OneHotEncoder, handle_unknown=ignore)을 통합하고 Pipeline으로 모델과 연결합니다.
5. 베이스라인 모델 비교: Logistic Regression, RandomForestClassifier, GradientBoosting 또는 XGBoost 중 2~3개를 동일한 CV 설정으로 비교해 Accuracy 평균/표준편차를 산출합니다.
6. 모델 선택 및 튜닝: 최고 성능 모델 1개를 선정해 핵심 하이퍼파라미터만 소규모 탐색하고(과도한 탐색 제외), CV Accuracy 개선 여부를 확인합니다.
7. 최종 학습 및 테스트 예측: 선택 모델을 churn_train 전체로 재학습 후 churn_test에 대해 예측을 생성하고, Customer_ID + 예측값 형태 결과를 저장합니다.
8. 사후 점검 및 리포트 정리: churn_test의 실제 Churn은 모델 선택에는 쓰지 않고 최종 점검용으로만 비교해 Accuracy, confusion matrix, classification_report를 기록합니다.

**Verification**
- 노트북 셀 실행 검증: 데이터 로드, 전처리, CV, 최종 예측까지 오류 없이 순차 실행 확인.
- 성능 검증: 5-Fold CV Accuracy 평균과 분산을 기록하고, 최종 선택 모델이 베이스라인 대비 개선됐는지 확인.
- 산출물 검증: 결과 파일 컬럼 형식(Customer_ID, Churn_pred 또는 과제 지정명)과 행 수가 [data/churn_test.csv](data/churn_test.csv)와 일치하는지 확인.

**Decisions**
- 1순위 평가지표: Accuracy
- 검증 전략: Stratified 5-Fold CV
- 테스트 데이터 사용 원칙: [data/churn_test.csv](data/churn_test.csv)의 Churn은 모델 선택에 미사용, 최종 사후 평가 전용
